# TTS Integration

Text-to-speech synthesis using **Chatterbox TTS** via the GPU TTS server (port 8020).
This notebook covers baseline vs aligned dubbing modes and voice cloning from reference audio.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

# Load .env (LOGFIRE_TOKEN, HF_TOKEN, etc.)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from foreign_whispers.client import FWClient, BASELINE, ALIGNED

fw = FWClient("http://localhost:8080")
fw.healthz()

# Optional: Logfire tracing (no-op shim if unavailable)
try:
    import logfire
    logfire.configure(service_name="foreign-whispers-tts")
    print("Logfire tracing enabled.")
except Exception:
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass
    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass
    logfire = _noop()
    print("Logfire not configured — using no-op shim.")

## Baseline TTS (No Alignment)

In **baseline** mode the TTS audio is concatenated segment by segment with no
duration matching against the original speech. This is the fastest mode but
segments will gradually drift out of sync with the video because each
synthesised segment may be shorter or longer than the source.

In [ ]:
video_id = "GYQ5yGV_-Oc"  # change to your video

with logfire.span("tts.baseline", video_id=video_id):
    baseline_result = fw.tts(video_id, config=BASELINE, alignment=False)

print(f"Audio path: {baseline_result['audio_path']}")
print(f"Config:     {baseline_result['config']}")

## Aligned TTS

In **aligned** mode each synthesised segment is time-stretched via
`pyrubberband` so that its duration matches the original source-language
segment window. This is slower but keeps the dubbed audio synchronised
with the video.

In [ ]:
with logfire.span("tts.aligned", video_id=video_id):
    aligned_result = fw.tts(video_id, config=ALIGNED, alignment=True)

print(f"Audio path: {aligned_result['audio_path']}")
print(f"Config:     {aligned_result['config']}")

## Compare Audio Durations

Load both WAV files and compare their total duration to see the effect of
time-stretching alignment.

In [ ]:
tts_root = PROJECT_ROOT / "pipeline_data" / "api" / "tts_audio" / "chatterbox"

baseline_dir = tts_root / BASELINE
aligned_dir = tts_root / ALIGNED


def wav_duration(wav_path: Path) -> float:
    """Return duration in seconds. Uses scipy if available, else file-size estimate."""
    try:
        from scipy.io import wavfile

        sr, data = wavfile.read(wav_path)
        return len(data) / sr
    except ImportError:
        # rough estimate: assume 16-bit mono 22050 Hz
        size = wav_path.stat().st_size - 44  # subtract WAV header
        return size / (22050 * 2)


for label, d in [("Baseline", baseline_dir), ("Aligned", aligned_dir)]:
    wavs = sorted(d.glob(f"{video_id}*.wav")) if d.exists() else []
    if wavs:
        dur = wav_duration(wavs[0])
        print(f"{label:10s}  {wavs[0].name}  duration={dur:.2f}s")
    else:
        print(f"{label:10s}  no WAV found in {d}")

## Speaker Reference Voices

Chatterbox supports voice cloning from short reference WAV clips. Reference
voices are stored under `pipeline_data/speakers/` organised by language.

In [ ]:
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"

if speakers_dir.exists():
    for lang_dir in sorted(speakers_dir.iterdir()):
        if lang_dir.is_dir():
            wavs = sorted(lang_dir.glob("*.wav"))
            print(f"{lang_dir.name}/  ({len(wavs)} WAV files)")
            for w in wavs:
                size_kb = w.stat().st_size / 1024
                print(f"  {w.name}  ({size_kb:.1f} KB)")
else:
    print(f"Speakers directory not found: {speakers_dir}")

---

## Voice Cloning Integration

The Chatterbox container supports voice cloning — it accepts a reference audio file via the `/v1/audio/speech/upload` endpoint for voice matching. However, the Foreign Whispers pipeline currently **hardcodes** `default.wav` and never exposes speaker selection through the API.

Your task is to wire voice cloning end-to-end so that:
1. The API accepts a `speaker_wav` parameter
2. Language-specific reference voices are selected automatically
3. (If diarization ran) different speakers use different reference voices

### Current State

| Component | Status |
| --------- | ------ |
| `tts.py` → `ChatterboxClient` | Supports `speaker_wav` kwarg per call |
| `CHATTERBOX_SPEAKER_WAV` env var | Defaults to empty string |
| `api/src/routers/tts.py` | No speaker parameter |
| `api/src/services/tts_service.py` | No speaker passthrough |
| `pipeline_data/speakers/{lang}/` | Reference WAVs exist but unused |
| Docker volume mount | `./pipeline_data/speakers:/app/voices` mounted |

### Pipeline Architecture

```
API endpoint                    tts.py                        Chatterbox Container
POST /api/tts/{video_id}  →  ChatterboxClient.tts_to_file()  →  POST /v1/audio/speech/upload
   ?speaker_wav=es/default.wav     speaker_wav=...                voice_file=...
   (YOUR WORK)                     (already supported)            (already supported)
```

### Task 1: Understand the Existing Chatterbox Client

Read the code that already handles speaker_wav to understand what's already wired.

In [ ]:
# Read the relevant sections of tts.py
tts_path = PROJECT_ROOT / "tts.py"
lines = tts_path.read_text().splitlines()

print("=== Chatterbox Configuration (lines 15-18) ===")
for line in lines[14:18]:
    print(f"  {line}")

print("\n=== ChatterboxClient constructor (lines 34-44) ===")
for line in lines[33:44]:
    print(f"  {line}")

print("\n=== tts_to_file method (lines 46-75) ===")
for line in lines[45:75]:
    print(f"  {line}")

print("\n--- Key observation ---")
print("ChatterboxClient.tts_to_file() already accepts **kwargs including 'speaker_wav'.")
print("The Chatterbox container receives a voice_file upload via /v1/audio/speech/upload.")
print("But nothing in the API layer passes this parameter through.")

In [ ]:
# Explore what reference voices are available
speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"

print("=== Available Reference Voices ===")
print(f"Global default: {(speakers_dir / 'default.wav').exists()}")
print()
for lang_dir in sorted(speakers_dir.iterdir()):
    if lang_dir.is_dir():
        wavs = sorted(lang_dir.glob("*.wav"))
        if wavs:
            print(f"{lang_dir.name}/")
            for w in wavs:
                size_kb = w.stat().st_size / 1024
                print(f"  {w.name}  ({size_kb:.1f} KB)")
        else:
            print(f"{lang_dir.name}/  (empty — needs reference WAVs)")

---

### Task 2: Voice Resolution Function

Write a pure function that resolves a speaker reference WAV path given a target language and optional speaker ID. This function will be used by the API to determine which voice file to pass to Chatterbox.

**File to create:** `foreign_whispers/voice_resolution.py`

**Resolution order:**
1. If speaker-specific WAV exists: `speakers/{lang}/{speaker_id}.wav`
2. If language default exists: `speakers/{lang}/default.wav`
3. Fall back to global: `speakers/default.wav`

#### 2.1 — Write the tests (TDD)

Run these tests first — they will fail because `resolve_speaker_wav` doesn't exist yet.

In [ ]:
# These tests define the contract for resolve_speaker_wav.
# DO NOT modify the tests — make your implementation pass them.

import tempfile
import os

def run_voice_tests():
    try:
        from foreign_whispers.voice_resolution import resolve_speaker_wav
    except (ImportError, ModuleNotFoundError):
        print("✗ foreign_whispers.voice_resolution not found — create the file first (Task 2.2)")
        return False

    passed, failed = 0, 0

    # Create a temporary speakers directory structure for testing
    with tempfile.TemporaryDirectory() as tmpdir:
        speakers = Path(tmpdir)

        # Create test structure:
        #   speakers/default.wav
        #   speakers/es/default.wav
        #   speakers/es/SPEAKER_00.wav
        #   speakers/fr/  (empty — no WAVs)
        (speakers / "default.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "es").mkdir()
        (speakers / "es" / "default.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "es" / "SPEAKER_00.wav").write_bytes(b"RIFF" + b"\x00" * 40)
        (speakers / "fr").mkdir()

        # Test 1: Speaker-specific WAV exists
        try:
            result = resolve_speaker_wav(speakers, "es", "SPEAKER_00")
            assert result == "es/SPEAKER_00.wav", f"Expected 'es/SPEAKER_00.wav', got '{result}'"
            print("✓ Test 1 passed: speaker-specific WAV")
            passed += 1
        except Exception as e:
            print(f"✗ Test 1 FAILED: {e}")
            failed += 1

        # Test 2: No speaker-specific WAV, falls back to language default
        try:
            result = resolve_speaker_wav(speakers, "es", "SPEAKER_01")
            assert result == "es/default.wav", f"Expected 'es/default.wav', got '{result}'"
            print("✓ Test 2 passed: language default fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 2 FAILED: {e}")
            failed += 1

        # Test 3: No language dir WAVs, falls back to global default
        try:
            result = resolve_speaker_wav(speakers, "fr", "SPEAKER_00")
            assert result == "default.wav", f"Expected 'default.wav', got '{result}'"
            print("✓ Test 3 passed: global default fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 3 FAILED: {e}")
            failed += 1

        # Test 4: No speaker_id provided, uses language default
        try:
            result = resolve_speaker_wav(speakers, "es")
            assert result == "es/default.wav", f"Expected 'es/default.wav', got '{result}'"
            print("✓ Test 4 passed: no speaker_id uses language default")
            passed += 1
        except Exception as e:
            print(f"✗ Test 4 FAILED: {e}")
            failed += 1

        # Test 5: Unknown language, falls back to global default
        try:
            result = resolve_speaker_wav(speakers, "xx")
            assert result == "default.wav", f"Expected 'default.wav', got '{result}'"
            print("✓ Test 5 passed: unknown language fallback")
            passed += 1
        except Exception as e:
            print(f"✗ Test 5 FAILED: {e}")
            failed += 1

    print(f"\n{'='*40}")
    print(f"Results: {passed} passed, {failed} failed")
    return failed == 0

# Run — expected to FAIL at this point
run_voice_tests()

#### 2.2 — Implement `resolve_speaker_wav`

Create `foreign_whispers/voice_resolution.py` with the stub below. Replace the `raise NotImplementedError` with your implementation.

**Hints:**
1. The return value is a **relative path** (e.g. `"es/SPEAKER_00.wav"`) — this is what the Chatterbox container expects relative to `/app/voices/`
2. Check paths in order: speaker-specific → language default → global default
3. Use `Path.exists()` to test each candidate

In [ ]:
# Stub — creates foreign_whispers/voice_resolution.py if it doesn't exist yet

voice_path = PROJECT_ROOT / "foreign_whispers" / "voice_resolution.py"

if voice_path.exists():
    print(f"File already exists: {voice_path}")
    print("Delete it manually if you want to start fresh.")
else:
    stub_code = '''\
"""Voice resolution for Chatterbox speaker cloning.

Resolves which reference WAV to use for a given target language
and optional speaker ID. The Chatterbox container expects a filename
relative to its /app/voices/ mount point.
"""

from pathlib import Path


def resolve_speaker_wav(
    speakers_dir: Path,
    target_language: str,
    speaker_id: str | None = None,
) -> str:
    """Resolve the reference WAV path for voice cloning.

    Resolution order:
    1. speakers/{lang}/{speaker_id}.wav  (if speaker_id given and file exists)
    2. speakers/{lang}/default.wav       (language-specific default)
    3. speakers/default.wav              (global fallback)

    Args:
        speakers_dir: Absolute path to the speakers directory.
        target_language: Language code (e.g. "es", "fr").
        speaker_id: Optional speaker identifier (e.g. "SPEAKER_00").

    Returns:
        Relative path string for the Chatterbox container (e.g. "es/default.wav").
    """
    # ---- YOUR CODE HERE ----
    raise NotImplementedError("Implement this function")
    # ---- END YOUR CODE ----
'''
    voice_path.write_text(stub_code)
    print(f"Created stub: {voice_path}")

#### 2.3 — Re-run the tests

After implementing `resolve_speaker_wav`, re-run the tests. All 5 should pass.

In [ ]:
# Reload and re-run (only works after you've implemented the function)
try:
    import importlib
    import foreign_whispers.voice_resolution
    importlib.reload(foreign_whispers.voice_resolution)
    if run_voice_tests():
        print("\nAll tests passed!")
    else:
        print("\nSome tests failed — fix your implementation before continuing.")
except (ImportError, ModuleNotFoundError):
    print("Skipped — create foreign_whispers/voice_resolution.py first (Task 2.2)")

#### 2.4 — Commit

```bash
git add foreign_whispers/voice_resolution.py
git commit -m "feat: add resolve_speaker_wav voice resolution function"
```

---

### Task 3: Add `speaker_wav` Parameter to the TTS API

**Goal:** Expose speaker selection through the API so callers can choose a reference voice.

**Files to modify:**
- `api/src/core/config.py` — add `speakers_dir` property
- `api/src/routers/tts.py` — add `speaker_wav` query parameter
- `api/src/services/tts_service.py` — pass `speaker_wav` through to `tts.py`

#### 3.1 — Add `speakers_dir` to Settings

Open `api/src/core/config.py` and add this property:

```python
@property
def speakers_dir(self) -> Path:
    return self.base_dir / "pipeline_data" / "speakers"
```

#### 3.2 — Add `speaker_wav` to the TTS endpoint

Open `api/src/routers/tts.py`. Add a new query parameter and pass it through:

```python
from foreign_whispers.voice_resolution import resolve_speaker_wav

@router.post("/tts/{video_id}")
async def tts_endpoint(
    video_id: str,
    request: Request,
    config: str = Query(..., pattern=r"^c-[0-9a-f]{7}$"),
    alignment: bool = Query(False),
    speaker_wav: str = Query(None, description="Reference voice WAV path (e.g. 'es/default.wav')"),
):
```

Inside the endpoint, if `speaker_wav` is not provided, resolve it automatically:

```python
if speaker_wav is None:
    speaker_wav = resolve_speaker_wav(settings.speakers_dir, "es")
```

#### 3.3 — Pass `speaker_wav` through the service

Open `api/src/services/tts_service.py`. Modify `text_file_to_speech` to accept and forward `speaker_wav`:

```python
def text_file_to_speech(self, source_path: str, output_path: str,
                        *, alignment: bool | None = None,
                        speaker_wav: str | None = None) -> None:
    tts_text_file_to_speech(source_path, output_path, self.tts_engine,
                            alignment=alignment, speaker_wav=speaker_wav)
```

Then in `tts.py`, the `text_file_to_speech` function must pass `speaker_wav` as a kwarg to `ChatterboxClient.tts_to_file()`. Find where `tts_to_file` is called and add:

```python
client.tts_to_file(text, wav_path, speaker_wav=speaker_wav)
```

#### 3.4 — Test manually

In [ ]:
# Rebuild and restart the API (run manually after implementing Task 3)
# !cd {PROJECT_ROOT} && docker compose --profile nvidia build api
# !cd {PROJECT_ROOT} && docker compose --profile nvidia up -d api
print("Uncomment the lines above and run this cell after implementing Task 3.")

In [ ]:
import requests

API_BASE = "http://localhost:8080"

# Test TTS with explicit speaker_wav (only works after Task 3 is implemented)
try:
    resp = requests.post(
        f"{API_BASE}/api/tts/{video_id}",
        params={"config": BASELINE, "alignment": "false", "speaker_wav": "es/default.wav"},
        timeout=10,
    )
    print(f"Status: {resp.status_code}")
    if resp.ok:
        print(resp.json())
    else:
        print(f"Error: {resp.text}")
except requests.ConnectionError:
    print("API not reachable — rebuild and restart after implementing Task 3.")

#### 3.5 — Commit

```bash
git add api/src/core/config.py api/src/routers/tts.py api/src/services/tts_service.py tts.py
git commit -m "feat: add speaker_wav parameter to TTS API endpoint"
```

---

### Task 4: Per-Speaker Voice Assignment

**Goal:** When transcription segments have `speaker` labels (from diarization), automatically assign different reference voices to different speakers.

**Prerequisite:** Task 5 from the `diarization_integration` notebook (segments have `speaker` field).

**File to modify:** `api/src/routers/tts.py`

#### 4.1 — Build a speaker-to-voice mapping

In the TTS endpoint, before calling TTS, read the translated transcript and extract unique speakers. Map each speaker to a reference WAV using `resolve_speaker_wav`:

```python
import json

# Load translated transcript to get speaker labels
trans_path = settings.translations_dir / f"{title}.json"
translated = json.loads(trans_path.read_text())
segments = translated.get("segments", [])

# Build speaker → voice mapping
unique_speakers = sorted(set(seg.get("speaker", "SPEAKER_00") for seg in segments))
voice_map = {
    spk: resolve_speaker_wav(settings.speakers_dir, "es", spk)
    for spk in unique_speakers
}
```

#### 4.2 — Modify `tts.py` to accept per-segment speaker hints

Currently `text_file_to_speech` synthesizes all segments with one `speaker_wav`. You need to modify it so that each segment can use a different speaker_wav based on its `speaker` field.

**Approach:** Pass `voice_map` as a dict to `text_file_to_speech`. Inside the function, for each segment, look up `voice_map[segment["speaker"]]` and pass it as `speaker_wav` to `tts_to_file()`.

This is the most open-ended part of the assignment. Study how `text_file_to_speech` iterates over segments in `tts.py` and decide where to inject the per-segment voice selection.

#### 4.3 — Test with a multi-speaker video

After implementing per-speaker voice assignment:

1. Run the diarization integration first to label segments with speaker IDs
2. Then run TTS — each speaker should use a different reference voice
3. Listen to the output to verify voice switching works

In [ ]:
# Verify voice mapping (after diarization has run)
import json

trans_dir = PROJECT_ROOT / "pipeline_data" / "api" / "translations" / "argos"
trans_files = list(trans_dir.glob("*.json"))

if trans_files:
    translated = json.loads(trans_files[0].read_text())
    segments = translated.get("segments", [])

    # Check which segments have speaker labels
    speakers = set()
    labeled = 0
    for seg in segments:
        spk = seg.get("speaker")
        if spk:
            speakers.add(spk)
            labeled += 1

    print(f"Total segments: {len(segments)}")
    print(f"With speaker labels: {labeled}")
    print(f"Unique speakers: {speakers or '(none — run diarization first)'}")

    if speakers:
        try:
            from foreign_whispers.voice_resolution import resolve_speaker_wav
            speakers_dir = PROJECT_ROOT / "pipeline_data" / "speakers"
            print("\nVoice mapping:")
            for spk in sorted(speakers):
                voice = resolve_speaker_wav(speakers_dir, "es", spk)
                print(f"  {spk} → {voice}")
        except (ImportError, ModuleNotFoundError):
            print("\nSkipped voice mapping — implement resolve_speaker_wav first (Task 2)")
else:
    print("No translated transcripts found — run the pipeline first.")

#### 4.4 — Commit

```bash
git add api/src/routers/tts.py api/src/services/tts_service.py tts.py
git commit -m "feat: per-speaker voice assignment in TTS"
```

---

## Evaluation Criteria

| # | Criterion | How to verify |
| - | --------- | ------------- |
| 1 | Tests pass | Re-run voice resolution tests — all 5 green |
| 2 | API accepts `speaker_wav` | `POST /api/tts/{video_id}?speaker_wav=es/default.wav` works |
| 3 | Auto-resolution works | Omitting `speaker_wav` selects language default automatically |
| 4 | Per-speaker mapping | With diarized segments, different speakers get different voices |
| 5 | Fallback chain | Unknown speaker/language falls back to `default.wav` |
| 6 | Code quality | Follows existing patterns (query params, service layer, config properties) |